# ChemicalEntity ↔ Disease Relation-Wise Merge

Merges Chemical–Disease triples from Monarch, DRKG, CKG, PharmKG, Hetionet, TARKG,
iBKH, PhytoChem, and EvoAGE; resolves chemical names via PubChem/DrugBank and disease
names via MESH/DO; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [6]:
import pandas as pd
import numpy as np
import re

# ── Base directories ──────────────────────────────────────────────────────────
BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'
EVOAGE_DIR = PROC_DIR + '4EvoAGE/'

# ── Output path ───────────────────────────────────────────────────────────────
OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/CHEMICALENTITY_DISEASE/ALL_CHEMICALENTITY_DISEASE.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. Chemical Lookups — HMDB, PubChem, DrugBank

In [2]:
# HMDB accession → PubChem CID
HMDB_2_pubchem = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/hmdb/HMDB_withname_pubchem.csv')
HMDB_2_pubchem['PUBCHEM_ID'] = HMDB_2_pubchem['PUBCHEM_ID'].astype(str).str.replace(r'\.0$', '', regex=True)
HMDB_2_pubchemDict = dict(zip(HMDB_2_pubchem['accession'], HMDB_2_pubchem['PUBCHEM_ID']))

print(f"HMDB → PubChem dict: {len(HMDB_2_pubchemDict):,} entries")

HMDB → PubChem dict: 217,920 entries


In [3]:
# PubChem: CID → IUPAC name and SMILES
Pubchem = pd.read_pickle(BASE_DIR + 'data_collection/databases_for_mapping/pubchem/combined_df.pkl')
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
del Pubchem

print(f"PubChem IUPAC dict:  {len(Pubchem_IUPAC_CID_Dict):,} entries")

PubChem IUPAC dict:  119,177,440 entries


In [4]:
# DrugBank: DrugBank ID → drug name
Drugbank = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/drugbank/ALL_DRUGBANK_WITH_SMILE_PUBCHEM.csv')
Drugbank_dict = dict(zip(Drugbank['drugbank_id'], Drugbank['name']))

print(f"DrugBank dict:       {len(Drugbank_dict):,} entries")

DrugBank dict:       16,575 entries


### 1b. Disease Lookups — DO, MESH, MedGen

In [5]:
# Disease Ontology: DOID → label and reverse
DO_All_File = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))

print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [6]:
# MedGen: source_id → preferred name; MONDO → MESH CUI mapping
MedGen_CUID_Source_ID = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MONDO_2_MESH    = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict       = dict(zip(MONDO_2_MESH['source_id'],       MONDO_2_MESH['#CUI_or_CN_id']))
MONDOMESH_2_des_dict    = dict(zip(MONDO_2_MESH['#CUI_or_CN_id'],   MONDO_2_MESH['pref_name']))

print(f"MedGen dict:   {len(MedGen_MeshID_name_dict):,} entries")
print(f"MONDO→MESH:    {len(MONDO_2_MESH_dict):,} entries")

MedGen dict:   394,895 entries
MONDO→MESH:    22,703 entries


In [7]:
# MESH xref → DOID mapping (exploded from TSV)
Mesh2DOID = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

print(f"MESH→DOID dict: {len(Mesh2DOID_dict):,} entries")

MESH→DOID dict: 3,687 entries


In [8]:
# MESH combined: ID → Name
MESH = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/MESH/MESH_Combined.csv')
MESH_dict      = dict(zip(MESH['ID'], MESH['Name']))
MESH_name_dict = MESH_dict  # alias for clarity

# MESH supplementary: ID → Name
Mesh_supp = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))

print(f"MESH dict:      {len(MESH_dict):,} entries")
print(f"MESH supp dict: {len(Mesh_supp_dict):,} entries")

MESH dict:      38,520 entries
MESH supp dict: 324,150 entries


In [9]:
MONDO = pd.read_csv(BASE_DIR + 'data_collection/databases_for_mapping/DO/MONDO_ID_Name_coll_from_Monarch.csv')
MONDO
MONDO_dict = dict(zip(MONDO['NAME'],MONDO['ID']))
MONDO_dict

{'colorectal Kaposi sarcoma': 'MONDO:0024659',
 'tubular adenoma': 'MONDO:0024660',
 'tubulovillous adenoma': 'MONDO:0024661',
 'colorectal tubulovillous adenoma': 'MONDO:0024662',
 'primary skin meningioma': 'MONDO:0024663',
 'hypertension, pregnancy-induced': 'MONDO:0024664',
 'indeterminate sex and/or pseudohermaphroditism': 'MONDO:0024665',
 'benign epithelial skin neoplasm': 'MONDO:0024666',
 'skin lymphangioma': 'MONDO:0024673',
 'Pancoast syndrome': 'MONDO:0024674',
 'adult kidney Wilms tumor': 'MONDO:0024675',
 'childhood kidney Wilms tumor': 'MONDO:0024676',
 'pancreatic insulinoma': 'MONDO:0024677',
 'Philadelphia-positive myelogenous leukemia': 'MONDO:0024685',
 'tenosynovial giant cell tumor, diffuse type': 'MONDO:0024686',
 'malignant mixed epithelial stromal tumor of the kidney': 'MONDO:0024711',
 'benign synovial neoplasm': 'MONDO:0024715',
 'childhood choroid plexus neoplasm': 'MONDO:0024744',
 'immature teratoma': 'MONDO:0024746',
 'cardiovascular neoplasm': 'MONDO:002

## 2. Load KG Sources

### 2a. ttd

,head,relation,relation.1,tail,head_type,head_type.1,tail_type,tail_type.1,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source
0,Curcuma longa (turmeric),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,"2-methyl-6-(4-methylcyclohexa-2,4-dien-1-yl)he...",PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,NaN,NaN,https://github.com/asgarhussain/phytochemicals...
1,Solanum torvum (Pea Eggplant),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,glycoalkaloid,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,NaN,NaN,https://github.com/asgarhussain/phytochemicals...
2,Rubia cordifolia(Indian Madder),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,124219,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"methyl 6-hydroxy-2,2-dimethylbenzo[h]chromene-...",NaN,Pubchem,https://github.com/asgarhussain/phytochemicals...
3,NaN,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,3-phenyl-7-[2-(piperidin-1-yl)ethoxy]-4-{4-[2-...,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,NaN,NaN,https://github.com/asgarhussain/phytochemicals...
4,Olea europaea (Olive),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,11652416,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,"2-(4-hydroxyphenyl)ethyl (E,3S)-4-formyl-3-(2-...",NaN,Pubchem,https://github.com/asgarhussain/phytochemicals...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,Diospyros lotus (Date Plum),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,extract,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,NaN,NaN,https://github.com/asgarhussain/phytochemicals...
876,Dracocephalum surmandinum,PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,essential oil,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,NaN,NaN,https://github.com/asgarhussain/phytochemicals...
877,Arabidopsis thaliana (Thale Cress),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,864,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,5-(dithiolan-3-yl)pentanoic acid,NaN,Pubchem,https://github.com/asgarhussain/phytochemicals...
878,Ferula assa-foetida (Asafoetida),PlantSpecies_Chemical,PlantSpecies_ChemicalEntity,essential oil,PlantSpecies,PlantSpecies,Chemical,ChemicalEntity,NaN,NaN,NaN,NaN,https://github.com/asgarhussain/phytochemicals...


In [10]:
Monarch_Chemical_Disease = pd.read_csv(PROC_DIR + 'Monarch/Human/Human_ChemicalEntity_Disease_Monarch.csv')

Monarch_Chemical_Disease['Tail_DO_name'] = (
    Monarch_Chemical_Disease['Tail_name']
    .map(DOID_disname_DOID_dict)
    .fillna(Monarch_Chemical_Disease['Tail_name'].map(MONDO_dict))
    .fillna(Monarch_Chemical_Disease['Tail'])
)


Monarch_Chemical_Disease[['Tail', 'Tail_DO_name']] = Monarch_Chemical_Disease[['Tail_DO_name', 'Tail']]
Monarch_Chemical_Disease['Tail_ID_IS'] = np.where(
    Monarch_Chemical_Disease['Tail'].isna(),
    None,
    np.where(
        Monarch_Chemical_Disease['Tail'].str.startswith('DOID'),
        'DOID',
        'MONDO'
    )
)


Monarch_Chemical_Disease.columns = Monarch_Chemical_Disease.columns.str.lower()
Monarch_Chemical_Disease

Monarch_Chemical_Disease.rename(columns={'kgsource': 'kg_source', 'head_name': 'head_detail_name', 'tail_name': 'tail_detail_name'}, inplace=True)

Monarch_Chemical_Disease['kg_type'] = 'Generalised'
Monarch_Chemical_Disease['kg_source'] = 'Monarch_KG'
Monarch_Chemical_Disease['species'] = 'Homo sapiens'


print(f"Monarch:  {len(Monarch_Chemical_Disease):,} rows | columns: {list(Monarch_Chemical_Disease.columns)}")
Monarch_Chemical_Disease

Monarch:  6,789 rows | columns: ['head', 'tail', 'relation_type', 'relation_source', 'kg_source', 'head_detail_name', 'head_type', 'head_id_is', 'head_taxon', 'head_taxon_label', 'tail_detail_name', 'tail_type', 'tail_id_is', 'tail_taxon', 'tail_taxon_label', 'head_taxon_name', 'tail_taxon_name', 'head_type_clean', 'tail_type_clean', 'relation', 'species_species', 'head_do_name', 'head_id', 'tail_do_name', 'kg_type', 'species']


,head,tail,relation_type,relation_source,kg_source,head_detail_name,head_type,head_id_is,head_taxon,head_taxon_label,...,tail_taxon_name,head_type_clean,tail_type_clean,relation,species_species,head_do_name,head_id,tail_do_name,kg_type,species
0,126517,DOID:552,treats_or_applied_or_studied_to_treat,infores:ctd,Monarch_KG,"1,5-anhydro-D-fructose",ChemicalEntity,MONDO,NaN,NaN,...,NaN,ChemicalEntity,Disease,ChemicalEntity_Disease,NaN,CHEBI:16715,CHEBI:16715,MONDO:0005249,Generalised,Homo sapiens
1,263,DOID:10808,treats_or_applied_or_studied_to_treat,infores:ctd,Monarch_KG,butan-1-ol,ChemicalEntity,MONDO,NaN,NaN,...,NaN,ChemicalEntity,Disease,ChemicalEntity_Disease,NaN,CHEBI:28885,CHEBI:28885,MONDO:0001126,Generalised,Homo sapiens
2,19,DOID:3021,treats_or_applied_or_studied_to_treat,infores:ctd,Monarch_KG,"2,3-dihydroxybenzoic acid",ChemicalEntity,MONDO,NaN,NaN,...,NaN,ChemicalEntity,Disease,ChemicalEntity_Disease,NaN,CHEBI:18026,CHEBI:18026,MONDO:0002492,Generalised,Homo sapiens
3,19,MONDO:0005365,treats_or_applied_or_studied_to_treat,infores:ctd,Monarch_KG,"2,3-dihydroxybenzoic acid",ChemicalEntity,MONDO,NaN,NaN,...,NaN,ChemicalEntity,Disease,ChemicalEntity_Disease,NaN,CHEBI:18026,CHEBI:18026,MONDO:0005365,Generalised,Homo sapiens
4,19,MONDO:0005240,treats_or_applied_or_studied_to_treat,infores:ctd,Monarch_KG,"2,3-dihydroxybenzoic acid",ChemicalEntity,MONDO,NaN,NaN,...,NaN,ChemicalEntity,Disease,ChemicalEntity_Disease,NaN,CHEBI:18026,CHEBI:18026,MONDO:0005240,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6784,5734,MONDO:0021667,treats_or_applied_or_studied_to_treat,infores:ctd,Monarch_KG,zonisamide,ChemicalEntity,MONDO,NaN,NaN,...,NaN,ChemicalEntity,Disease,ChemicalEntity_Disease,NaN,CHEBI:10127,CHEBI:10127,MONDO:0021667,Generalised,Homo sapiens
6785,5734,MONDO:0005180,treats_or_applied_or_studied_to_treat,infores:ctd,Monarch_KG,zonisamide,ChemicalEntity,MONDO,NaN,NaN,...,NaN,ChemicalEntity,Disease,ChemicalEntity_Disease,NaN,CHEBI:10127,CHEBI:10127,MONDO:0005180,Generalised,Homo sapiens
6786,5734,DOID:11832,treats_or_applied_or_studied_to_treat,infores:ctd,Monarch_KG,zonisamide,ChemicalEntity,MONDO,NaN,NaN,...,NaN,ChemicalEntity,Disease,ChemicalEntity_Disease,NaN,CHEBI:10127,CHEBI:10127,MONDO:0001386,Generalised,Homo sapiens
6787,5734,DOID:1824,treats_or_applied_or_studied_to_treat,infores:ctd,Monarch_KG,zonisamide,ChemicalEntity,MONDO,NaN,NaN,...,NaN,ChemicalEntity,Disease,ChemicalEntity_Disease,NaN,CHEBI:10127,CHEBI:10127,MONDO:0002125,Generalised,Homo sapiens


In [11]:
Monarch_Chemical_Disease['tail_id_is'].value_counts()

tail_id_is
DOID     3696
MONDO    3093
Name: count, dtype: int64

### 2b. DRKG

In [12]:
DRKG_Chemical_Disease = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Drug_Disease.csv')
DRKG_Chemical_Disease.columns = DRKG_Chemical_Disease.columns.str.lower()
print(f"DRKG:     {len(DRKG_Chemical_Disease):,} rows | columns: {list(DRKG_Chemical_Disease.columns)}")

DRKG_Chemical_Disease['kg_type'] = 'Generalised'
DRKG_Chemical_Disease['kg_source'] = 'DRKG'
DRKG_Chemical_Disease['species'] = 'Homo sapiens'

DRKG_Chemical_Disease.head(2)

DRKG:     58,032 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'head_orignal', 'tail_orignal']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id,tail_detail_name,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,DB00003,Compound_Disease,D003550,Compound,DRUGBANK::treats::Compound:Disease,Disease,DRKG,DB00003,Cystic Fibrosis,Drugbank,MESH,Compound::DB00003,Disease::MESH:D003550,Generalised,Homo sapiens
1,DB00004,Compound_Disease,C063419,Compound,DRUGBANK::treats::Compound:Disease,Disease,DRKG,DB00004,"CTAGE1 protein, human",Drugbank,MESH,Compound::DB00004,Disease::MESH:C063419,Generalised,Homo sapiens


### 2c. CKG (Metabolite–Disease)

Head IDs are HMDB accessions — map to PubChem CIDs before concat.

In [13]:
CKG_Chemical_Disease = pd.read_csv(PROC_DIR + 'CKG/File_31_Metabolite_Disease_CKG.csv')
CKG_Chemical_Disease.columns = CKG_Chemical_Disease.columns.str.lower()
# Map HMDB → PubChem CID; drop rows with no mapping
CKG_Chemical_Disease['head_id'] = CKG_Chemical_Disease['head'].map(HMDB_2_pubchemDict)
CKG_Chemical_Disease = CKG_Chemical_Disease[~CKG_Chemical_Disease['head_id'].isna()]
# Swap head ↔ head_id so canonical ID is in 'head'
CKG_Chemical_Disease[['head', 'head_id']] = CKG_Chemical_Disease[['head_id', 'head']]
CKG_Chemical_Disease.rename(columns={'head_iupac_name': 'head_detail_name', 'tail_do_name': 'tail_detail_name'}, inplace=True)
print(f"CKG:      {len(CKG_Chemical_Disease):,} rows | columns: {list(CKG_Chemical_Disease.columns)}")

CKG_Chemical_Disease['kg_type'] = 'Generalised'
CKG_Chemical_Disease['kg_source'] = 'CKG'
CKG_Chemical_Disease['species'] = 'Homo sapiens'


CKG_Chemical_Disease.head(2)

CKG:      24,747 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'source', 'kg_source', 'head_common_name', 'head_detail_name', 'tail_detail_name', 'tail_do_alt_ids', 'head_id_is', 'tail_id_is', 'head_id']


,head,relation,tail,head_type,tail_type,source,kg_source,head_common_name,head_detail_name,tail_detail_name,tail_do_alt_ids,head_id_is,tail_id_is,head_id,kg_type,species
0,92105,ASSOCIATED_WITH,610247,Metabolite,Disease,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,NaN,610247,HMDB,Disease_Ontology,HMDB0000001,Generalised,Homo sapiens
1,92105,ASSOCIATED_WITH,601665,Metabolite,Disease,HMDB,CKG,1-methylhistidine,(2S)-2-amino-3-(1-methyl-1H-imidazol-4-yl)prop...,NaN,601665,HMDB,Disease_Ontology,HMDB0000001,Generalised,Homo sapiens


### 2d. PharmKG

In [14]:
PharmKG_Chemical_Disease = pd.read_csv(PROC_DIR + 'PharmKG/PharmKG_Chemical_Disease.csv')
PharmKG_Chemical_Disease.columns = PharmKG_Chemical_Disease.columns.str.lower()
print(f"PharmKG:  {len(PharmKG_Chemical_Disease):,} rows | columns: {list(PharmKG_Chemical_Disease.columns)}")

PharmKG_Chemical_Disease['kg_type'] = 'Generalised'
PharmKG_Chemical_Disease['kg_source'] = 'PharmKG'
PharmKG_Chemical_Disease['species'] = 'Homo sapiens'

PharmKG_Chemical_Disease.head(2)

PharmKG:  70,487 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_orignal', 'tail_orignal', 'pubmed_id', 'sentence_tokenized']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_orignal,tail_orignal,pubmed_id,sentence_tokenized,kg_type,species
0,5280483,Chemical_Disease,DOID:1485,chemical,T,disease,PharmKG,Pubchem,DOID,vitamin k,cystic fibrosis,"nan, nan","nan, nan",Generalised,Homo sapiens
1,681,Chemical_Disease,DOID:13593,chemical,T,disease,PharmKG,Pubchem,DOID,dopamine,eclampsia,7261354.0,'Dopamine treatment for prevention of renal_fa...,Generalised,Homo sapiens


### 2e. Hetionet

In [15]:
Hetionet_Chemical_Disease = pd.read_csv(PROC_DIR + 'Hetionet/Compound_Disease_Hetionet.csv')
Hetionet_Chemical_Disease.columns = Hetionet_Chemical_Disease.columns.str.lower()
print(f"Hetionet: {len(Hetionet_Chemical_Disease):,} rows | columns: {list(Hetionet_Chemical_Disease.columns)}")


PharmKG_Chemical_Disease['kg_type'] = 'Generalised'
PharmKG_Chemical_Disease['kg_source'] = 'Hetionet'
PharmKG_Chemical_Disease['species'] = 'Homo sapiens'

Hetionet_Chemical_Disease.head(2)

Hetionet: 1,145 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_id_is', 'tail_id_is', 'head_pubchem_mapped', 'tail_name']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_pubchem_mapped,tail_name
0,31703,Compound_Disease,DOID:363,Compound,Compound–treats–Disease,Disease,Hetionet,Pubchem,DOID,DB00997,uterine cancer
1,5770,Compound_Disease,DOID:10763,Compound,Compound–treats–Disease,Disease,Hetionet,Pubchem,DOID,DB00206,hypertension


### 2f. TARKG

In [16]:
TARKG_Chemical_Disease = pd.read_csv(PROC_DIR + 'TARKG/Compound_Disease_TARKG.csv')
TARKG_Chemical_Disease.columns = TARKG_Chemical_Disease.columns.str.lower()
print(f"TARKG:    {len(TARKG_Chemical_Disease):,} rows | columns: {list(TARKG_Chemical_Disease.columns)}")


TARKG_Chemical_Disease['kg_type'] = 'Generalised'
TARKG_Chemical_Disease['kg_source'] = 'TARKG'
TARKG_Chemical_Disease['species'] = 'Homo sapiens'

TARKG_Chemical_Disease.head(2)

TARKG:    751,743 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_index', 'kg', 'change']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,4946,ChemicalEntity_Disease,DOID:175,ChemicalEntity,treat,Disease,TARKG,1-naphthalen-1-yloxy-3-(propan-2-ylamino)propa...,vascular cancer,Pubchem,DOID,addKG-5605254,addKG,0,Generalised,Homo sapiens
1,11349,ChemicalEntity_Disease,DOID:175,ChemicalEntity,cause,Disease,TARKG,3-hydroxy-2-phenylchromen-4-one,vascular cancer,Pubchem,DOID,addKG-11487153,addKG,0,Generalised,Homo sapiens


### 2g. iBKH

`tail_detail_name` and `tail_detail_name.1` are swapped in the raw file — corrected here.

In [17]:
iBKH_Chemical_Disease = pd.read_csv(PROC_DIR + 'iBKH/Chemical_Disease_ibkh.csv')
iBKH_Chemical_Disease.columns = iBKH_Chemical_Disease.columns.str.lower()

print(f"iBKH:     {len(iBKH_Chemical_Disease):,} rows | columns: {list(iBKH_Chemical_Disease.columns)}")


iBKH_Chemical_Disease['kg_type'] = 'Generalised'
iBKH_Chemical_Disease['kg_source'] = 'iBKH'
iBKH_Chemical_Disease['species'] = 'Homo sapiens'

iBKH_Chemical_Disease.head(2)

iBKH:     545,470 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,head_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,31703,ChemicalEntity_Disease,DOID:363,ChemicalEntity,Disease,iBKH,"(7S,9S)-7-[(2R,4S,5S,6S)-4-amino-5-hydroxy-6-m...",C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...,uterine cancer,Pubchem,DOID,Generalised,Homo sapiens
1,5770,ChemicalEntity_Disease,DOID:10763,ChemicalEntity,Disease,iBKH,"methyl (1R,15S,17R,18R,19S,20S)-6,18-dimethoxy...",CO[C@H]1[C@@H](C[C@@H]2CN3CCC4=C([C@H]3C[C@@H]...,hypertension,Pubchem,DOID,Generalised,Homo sapiens


### 2h. DTINet

In [18]:
dtinet_Chemical_Disease = pd.read_csv(PROC_DIR + 'DTINet/Chemical_Disease_DTINet.csv')

dtinet_Chemical_Disease.columns = dtinet_Chemical_Disease.columns.str.lower()

print(f"iBKH:     {len(dtinet_Chemical_Disease):,} rows | columns: {list(dtinet_Chemical_Disease.columns)}")

dtinet_Chemical_Disease['kg_type'] = 'Generalised'
dtinet_Chemical_Disease['kg_source'] = 'DtiNet'
dtinet_Chemical_Disease['species'] = 'Homo sapiens'

dtinet_Chemical_Disease

iBKH:     94,642 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_alt_id', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,25074887,ChemicalEntity_Disease,DOID:1596,ChemicalEntity,Disease,DtiNet,DB00050,(2S)-1-[(2S)-2-[[(2S)-2-[[(2R)-2-[[(2S)-2-[[(2...,depressive disorder,Pubchem,DOID,Generalised,Homo sapiens
1,25074887,ChemicalEntity_Disease,C0860207,ChemicalEntity,Disease,DtiNet,DB00050,(2S)-1-[(2S)-2-[[(2S)-2-[[(2R)-2-[[(2S)-2-[[(2...,drug-induced liver injury,Pubchem,MESH,Generalised,Homo sapiens
2,25074887,ChemicalEntity_Disease,C0027540,ChemicalEntity,Disease,DtiNet,DB00050,(2S)-1-[(2S)-2-[[(2S)-2-[[(2R)-2-[[(2S)-2-[[(2...,necrosis,Pubchem,MESH,Generalised,Homo sapiens
3,25074887,ChemicalEntity_Disease,DOID:12849,ChemicalEntity,Disease,DtiNet,DB00050,(2S)-1-[(2S)-2-[[(2S)-2-[[(2R)-2-[[(2S)-2-[[(2...,autistic disorder,Pubchem,DOID,Generalised,Homo sapiens
4,25074887,ChemicalEntity_Disease,DOID:10763,ChemicalEntity,Disease,DtiNet,DB00050,(2S)-1-[(2S)-2-[[(2S)-2-[[(2R)-2-[[(2S)-2-[[(2...,hypertension,Pubchem,DOID,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
94637,5282452,ChemicalEntity_Disease,DOID:9563,ChemicalEntity,Disease,DtiNet,DB08860,"(E,3R,5S)-7-[2-cyclopropyl-4-(4-fluorophenyl)q...",bronchiectasis,Pubchem,DOID,Generalised,Homo sapiens
94638,5282452,ChemicalEntity_Disease,C1834752,ChemicalEntity,Disease,DtiNet,DB08860,"(E,3R,5S)-7-[2-cyclopropyl-4-(4-fluorophenyl)q...","mycobacterium tuberculosis, susceptibility to",Pubchem,MESH,Generalised,Homo sapiens
94639,5282452,ChemicalEntity_Disease,C1855456,ChemicalEntity,Disease,DtiNet,DB08860,"(E,3R,5S)-7-[2-cyclopropyl-4-(4-fluorophenyl)q...",plasmodium falciparum blood infection level,Pubchem,MESH,Generalised,Homo sapiens
94640,5282452,ChemicalEntity_Disease,DOID:10327,ChemicalEntity,Disease,DtiNet,DB08860,"(E,3R,5S)-7-[2-cyclopropyl-4-(4-fluorophenyl)q...",anthracosis,Pubchem,DOID,Generalised,Homo sapiens


### 2h. PhytoChem

In [19]:
ttd_phytodata = pd.read_csv(PROC_DIR + 'ttd/Phytodata_TTD_Chemical_Disease.csv')
ttd_phytodata.columns = ttd_phytodata.columns.str.lower()
ttd_phytodata.rename(columns={'relation_source': 'kg_source'}, inplace=True)
print(f"PhytoChem:{len(ttd_phytodata):,} rows | columns: {list(ttd_phytodata.columns)}")

ttd_phytodata['kg_type'] = 'Generalised'
ttd_phytodata['kg_source'] = 'TTD'
ttd_phytodata['species'] = 'Homo sapiens'

ttd_phytodata.head(2)

PhytoChem:5,907 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_source']


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type,species
0,9831643,ChemicalEntity_Disease,D011537,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Pruritus,Pubchem,MESH_ID,TTD,Generalised,Homo sapiens
1,9831643,ChemicalEntity_Disease,C0268312,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Progressive familial intrahepatic cholestasis,Pubchem,MESH_ID,TTD,Generalised,Homo sapiens


In [20]:
ttd_phytodata_2 = pd.read_csv(PROC_DIR + 'ttd/Phytodata_Chemical_Disease.csv')
ttd_phytodata_2.columns = ttd_phytodata_2.columns.str.lower()
ttd_phytodata_2.rename(columns={'relation_source': 'kg_source'}, inplace=True)
print(f"PhytoChem:{len(ttd_phytodata_2):,} rows | columns: {list(ttd_phytodata_2.columns)}")

ttd_phytodata_2['kg_type'] = 'Generalised'
ttd_phytodata_2['species'] = 'Homo sapiens'

ttd_phytodata_2.head(2)

PhytoChem:1,448 rows | columns: ['head', 'relation', 'relation.1', 'tail', 'head_type', 'head_type.1', 'tail_type', 'tail_type.1', 'head_detail_name', 'tail_detail_name', 'kg_source']


,head,relation,relation.1,tail,head_type,head_type.1,tail_type,tail_type.1,head_detail_name,tail_detail_name,kg_source,kg_type,species
0,Alpha-Turmerone,Chemical_Disease,ChemicalEntity_Disease,DOID:1612,Chemical,ChemicalEntity,Disease,Disease,NaN,breast cancer,https://github.com/asgarhussain/phytochemicals...,Generalised,Homo sapiens
1,Extract of Solanum torvum,Chemical_Disease,ChemicalEntity_Disease,DOID:1612,Chemical,ChemicalEntity,Disease,Disease,NaN,breast cancer,https://github.com/asgarhussain/phytochemicals...,Generalised,Homo sapiens


### 2i. Crossbar

In [21]:
crossbar = pd.read_csv(PROC_DIR + 'crossbar/ChemicalEntity_Disease_Crossbar.csv')
crossbar.columns = crossbar.columns.str.lower()
crossbar.rename(columns={'relation_source': 'kg_source'}, inplace=True)
print(f"PhytoChem:{len(crossbar):,} rows | columns: {list(crossbar.columns)}")

crossbar['kg_type'] = 'Generalised'
crossbar['kg_source'] = 'CROssBAR'
crossbar['species'] = 'Homo sapiens'

crossbar.head(2)

PhytoChem:6 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'kg_source', 'head_alt_id', 'head_detail_name', 'tail_alt_id', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,tail_type,kg_source,head_alt_id,head_detail_name,tail_alt_id,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,5291,ChemicalEntity_Disease,DOID:0080638,ChemicalEntity,Disease,CROssBAR,DB00619,Imatinib,H00001,B-cell acute lymphoblastic leukemia,Pubchem,DOID,Generalised,Homo sapiens
1,441203,ChemicalEntity_Disease,DOID:8584,ChemicalEntity,Disease,CROssBAR,DB00515,Cisplatin,H00008,Burkitt lymphoma,Pubchem,DOID,Generalised,Homo sapiens


### 2i. hald

In [22]:
hald = pd.read_csv(PROC_DIR + 'hald/Chemical_Disease_HALD.csv')
hald.columns = hald.columns.str.lower()
hald.rename(columns={'relation_source': 'kg_source'}, inplace=True)
print(f"PhytoChem:{len(hald):,} rows | columns: {list(hald.columns)}")

hald['kg_type'] = 'Aging'
hald['kg_source'] = 'CROssBAR'
hald['species'] = 'Homo sapiens'


hald.head(2)

PhytoChem:4,330 rows | columns: ['head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type', 'kg_source', 'head_detail_name', 'head_smiles_name', 'tail_detail_name', 'head_id_is', 'tail_id_is']


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_smiles_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,31703,ChemicalEntity_Disease,DOID:3382,ChemicalEntity,stratify,Disease,CROssBAR,"(7S,9S)-7-[(2R,4S,5S,6S)-4-amino-5-hydroxy-6-m...",C[C@H]1[C@H]([C@H](C[C@@H](O1)O[C@H]2C[C@@](CC...,Liposarcoma,Pubchem,DOID,Aging,Homo sapiens
1,5793,ChemicalEntity_Disease,D009422,ChemicalEntity,prevent,Disease,CROssBAR,"(3R,4S,5S,6R)-6-(hydroxymethyl)oxane-2,3,4,5-t...",C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O,Nervous System Diseases,Pubchem,MESH,Aging,Homo sapiens


### 2j. pheknowlator

In [23]:
pheknowlator = pd.read_csv(PROC_DIR + 'pheknowlator/Chemical_Disease.csv')
pheknowlator.columns = pheknowlator.columns.str.lower()
pheknowlator.rename(columns={'relation_source': 'kg_source'}, inplace=True)
print(f"PhytoChem:{len(pheknowlator):,} rows | columns: {list(pheknowlator.columns)}")

pheknowlator['kg_type'] = 'Generalised'
pheknowlator['kg_source'] = 'pheknowlator'
pheknowlator['species'] = 'Homo sapiens'
pheknowlator.head(2)

PhytoChem:659,986 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'head_iupac', 'head_smiles', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_source']


,head,relation,tail,head_type,tail_type,head_iupac,head_smiles,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type,species
0,1108,Chemical_Disease,DOID:2799,Chemical,Disease,"2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17-hexade...",C1CCC2C(C1)CCC3C2CCC4C3CCC4,bronchiolitis obliterans,Pubchem,DOID,pheknowlator,Generalised,Homo sapiens
1,369312,Chemical_Disease,DOID:813,Chemical,Disease,[(4S)-4-prop-1-en-2-ylcyclohexen-1-yl]methanol,CC(=C)[C@H]1CCC(=CC1)CO,septic arthritis,Pubchem,DOID,pheknowlator,Generalised,Homo sapiens


### 2k. Plant sources (ttd)

In [32]:
ttd = pd.read_csv(PROC_DIR + 'ttd/Phytodata_TTD_Chemical_Disease.csv')
ttd.columns = ttd.columns.str.lower()
ttd.rename(columns={'relation_source': 'kg_source'}, inplace=True)
print(f"ttd:{len(ttd):,} rows | columns: {list(ttd.columns)}")
ttd['kg_type'] = 'Generalised'
ttd['kg_source'] = 'ttd'
ttd['species'] = 'Homo sapiens'
ttd

ttd:5,907 rows | columns: ['head', 'relation', 'tail', 'head_type', 'tail_type', 'head_detail_name', 'tail_detail_name', 'head_id_is', 'tail_id_is', 'kg_source']


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type,species
0,9831643,ChemicalEntity_Disease,D011537,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Pruritus,Pubchem,MESH_ID,ttd,Generalised,Homo sapiens
1,9831643,ChemicalEntity_Disease,C0268312,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Progressive familial intrahepatic cholestasis,Pubchem,MESH_ID,ttd,Generalised,Homo sapiens
2,9831643,ChemicalEntity_Disease,DOID:9245,ChemicalEntity,Disease,"(4R,5R)-5-[4-[[4-(4-aza-1-azoniabicyclo[2.2.2]...",Alagille syndrome,Pubchem,DOID_ID,ttd,Generalised,Homo sapiens
3,129269915,ChemicalEntity_Disease,C0024305,ChemicalEntity,Disease,5-amino-3-[4-[[(5-fluoro-2-methoxybenzoyl)amin...,Non-hodgkin lymphoma,Pubchem,MESH_ID,ttd,Generalised,Homo sapiens
4,77050711,ChemicalEntity_Disease,C0700345,ChemicalEntity,Disease,"(2R)-2-(2,4-difluorophenyl)-1,1-difluoro-3-(te...",Vulvovaginal Candidiasis,Pubchem,MESH_ID,ttd,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...
5902,54680316,ChemicalEntity_Disease,C0019693,ChemicalEntity,Disease,6-[(3-chloro-4-fluorophenyl)methyl]-4-hydroxy-...,Human immunodeficiency virus infection,Pubchem,MESH_ID,ttd,Generalised,Homo sapiens
5903,50992246,ChemicalEntity_Disease,D007249,ChemicalEntity,Disease,"[(1S,3R)-3-[[(3S,4S)-3-methoxyoxan-4-yl]amino]...",Inflammation,Pubchem,MESH_ID,ttd,Generalised,Homo sapiens
5904,57588528,ChemicalEntity_Disease,C0021390,ChemicalEntity,Disease,[4-(3-sulfanylidenedithiol-4-yl)phenyl] 5-amin...,Inflammatory bowel disease,Pubchem,MESH_ID,ttd,Generalised,Homo sapiens
5905,46183193,ChemicalEntity_Disease,C0003469,ChemicalEntity,Disease,"3-(2,5-difluorophenyl)-7-[1,1,1,3,3,3-hexadeut...",Anxiety disorder,Pubchem,MESH_ID,ttd,Generalised,Homo sapiens


In [3]:
#

## 3. Check and Fix Duplicate Columns

`pd.concat` raises `InvalidIndexError` if any DataFrame has duplicate column names.
Inspect here and keep the occurrence with real data.

In [33]:
SOURCE_DFS = [
    ('pheknowlator',pheknowlator),
    ('crossbar',   crossbar),
    ('hald',    hald),
    ('ttd_phytodata',   ttd_phytodata),
    ('ttd_phytodata_2',                   ttd_phytodata_2),
    ('dtinet_Chemical_Disease',                   dtinet_Chemical_Disease),
    ('iBKH_Chemical_Disease', iBKH_Chemical_Disease),
    ('TARKG_Chemical_Disease',   TARKG_Chemical_Disease),
    ('Hetionet_Chemical_Disease',    Hetionet_Chemical_Disease),
    ('PharmKG_Chemical_Disease',   PharmKG_Chemical_Disease),
    ('CKG_Chemical_Disease',                   CKG_Chemical_Disease),
    ('DRKG_Chemical_Disease',                   DRKG_Chemical_Disease),
    ('Monarch_Chemical_Disease',                   Monarch_Chemical_Disease),
    ('ttd',                   ttd),
    

]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            print(f"  '{col}':")
            for i, (_, series) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"    occurrence {i} → sample: {series.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicate columns")

[pheknowlator] ✓ no duplicate columns
[crossbar] ✓ no duplicate columns
[hald] ✓ no duplicate columns
[ttd_phytodata] ✓ no duplicate columns
[ttd_phytodata_2] ✓ no duplicate columns
[dtinet_Chemical_Disease] ✓ no duplicate columns
[iBKH_Chemical_Disease] ✓ no duplicate columns
[TARKG_Chemical_Disease] ✓ no duplicate columns
[Hetionet_Chemical_Disease] ✓ no duplicate columns
[PharmKG_Chemical_Disease] ✓ no duplicate columns
[CKG_Chemical_Disease] ✓ no duplicate columns
[DRKG_Chemical_Disease] ✓ no duplicate columns
[Monarch_Chemical_Disease] ✓ no duplicate columns
[ttd] ✓ no duplicate columns


## 4. Align Schemas and Concatenate

In [34]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type','species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df

Combined: 2,230,639 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,1108,Chemical_Disease,DOID:2799,Chemical,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,bronchiolitis obliterans,Generalised,Homo sapiens
1,369312,Chemical_Disease,DOID:813,Chemical,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,septic arthritis,Generalised,Homo sapiens
2,37439,Chemical_Disease,DOID:687,Chemical,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,hepatoblastoma,Generalised,Homo sapiens
3,338,Chemical_Disease,DOID:2352,Chemical,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,hemochromatosis,Generalised,Homo sapiens
4,3036,Chemical_Disease,DOID:350,Chemical,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,mastocytosis,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2230634,54680316,ChemicalEntity_Disease,C0019693,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH_ID,6-[(3-chloro-4-fluorophenyl)methyl]-4-hydroxy-...,Human immunodeficiency virus infection,Generalised,Homo sapiens
2230635,50992246,ChemicalEntity_Disease,D007249,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH_ID,"[(1S,3R)-3-[[(3S,4S)-3-methoxyoxan-4-yl]amino]...",Inflammation,Generalised,Homo sapiens
2230636,57588528,ChemicalEntity_Disease,C0021390,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH_ID,[4-(3-sulfanylidenedithiol-4-yl)phenyl] 5-amin...,Inflammatory bowel disease,Generalised,Homo sapiens
2230637,46183193,ChemicalEntity_Disease,C0003469,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH_ID,"3-(2,5-difluorophenyl)-7-[1,1,1,3,3,3-hexadeut...",Anxiety disorder,Generalised,Homo sapiens


## 5. Standardise Fixed-Value Columns

In [36]:
final_df['head_type'] = 'ChemicalEntity'
final_df['relation']  = 'ChemicalEntity_Disease'
final_df['tail_type'] = 'Disease'

# head_id_is: DB-prefix → DrugBank, else PubChem
final_df['head'] = final_df['head'].astype(str)
final_df['tail'] = final_df['tail'].astype(str)

final_df['head_id_is'] = np.where(
    final_df['head'].isna(), None,
    np.where(final_df['head'].str.startswith('DB'), 'Drugbank', 'Pubchem')
)

# tail_id_is: DOID-prefix → DO_ID, else MESH
final_df['tail_id_is'] = np.where(
    final_df['tail'].isna(), None,
    np.where(final_df['tail'].str.startswith('DOID'), 'DOID', 'MESH')
)

print("Unique relation:",   set(final_df['relation']))
print("Unique head_type:",  set(final_df['head_type']))
print("Unique tail_type:",  set(final_df['tail_type']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
final_df

Unique relation: {'ChemicalEntity_Disease'}
Unique head_type: {'ChemicalEntity'}
Unique tail_type: {'Disease'}
Unique head_id_is: {'Drugbank', 'Pubchem'}
Unique tail_id_is: {'MESH', 'DOID'}


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,1108,ChemicalEntity_Disease,DOID:2799,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,bronchiolitis obliterans,Generalised,Homo sapiens
1,369312,ChemicalEntity_Disease,DOID:813,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,septic arthritis,Generalised,Homo sapiens
2,37439,ChemicalEntity_Disease,DOID:687,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,hepatoblastoma,Generalised,Homo sapiens
3,338,ChemicalEntity_Disease,DOID:2352,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,hemochromatosis,Generalised,Homo sapiens
4,3036,ChemicalEntity_Disease,DOID:350,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,mastocytosis,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2230634,54680316,ChemicalEntity_Disease,C0019693,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH,6-[(3-chloro-4-fluorophenyl)methyl]-4-hydroxy-...,Human immunodeficiency virus infection,Generalised,Homo sapiens
2230635,50992246,ChemicalEntity_Disease,D007249,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH,"[(1S,3R)-3-[[(3S,4S)-3-methoxyoxan-4-yl]amino]...",Inflammation,Generalised,Homo sapiens
2230636,57588528,ChemicalEntity_Disease,C0021390,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH,[4-(3-sulfanylidenedithiol-4-yl)phenyl] 5-amin...,Inflammatory bowel disease,Generalised,Homo sapiens
2230637,46183193,ChemicalEntity_Disease,C0003469,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH,"3-(2,5-difluorophenyl)-7-[1,1,1,3,3,3-hexadeut...",Anxiety disorder,Generalised,Homo sapiens


## 6. Filter Valid Head IDs

Keep only rows where head is a DrugBank ID (`DB…`) or a numeric PubChem CID.

In [37]:
before = len(final_df)
final_df = final_df[
    final_df['head'].str.startswith('DB') | final_df['head'].str.isnumeric()
].reset_index(drop=True)
print(f"Filtered {before - len(final_df):,} rows with non-standard head IDs → {len(final_df):,} remaining")
final_df

Filtered 2,291 rows with non-standard head IDs → 2,228,348 remaining


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,1108,ChemicalEntity_Disease,DOID:2799,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,bronchiolitis obliterans,Generalised,Homo sapiens
1,369312,ChemicalEntity_Disease,DOID:813,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,septic arthritis,Generalised,Homo sapiens
2,37439,ChemicalEntity_Disease,DOID:687,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,hepatoblastoma,Generalised,Homo sapiens
3,338,ChemicalEntity_Disease,DOID:2352,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,hemochromatosis,Generalised,Homo sapiens
4,3036,ChemicalEntity_Disease,DOID:350,ChemicalEntity,NaN,Disease,pheknowlator,Pubchem,DOID,NaN,mastocytosis,Generalised,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2228343,54680316,ChemicalEntity_Disease,C0019693,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH,6-[(3-chloro-4-fluorophenyl)methyl]-4-hydroxy-...,Human immunodeficiency virus infection,Generalised,Homo sapiens
2228344,50992246,ChemicalEntity_Disease,D007249,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH,"[(1S,3R)-3-[[(3S,4S)-3-methoxyoxan-4-yl]amino]...",Inflammation,Generalised,Homo sapiens
2228345,57588528,ChemicalEntity_Disease,C0021390,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH,[4-(3-sulfanylidenedithiol-4-yl)phenyl] 5-amin...,Inflammatory bowel disease,Generalised,Homo sapiens
2228346,46183193,ChemicalEntity_Disease,C0003469,ChemicalEntity,NaN,Disease,ttd,Pubchem,MESH,"3-(2,5-difluorophenyl)-7-[1,1,1,3,3,3-hexadeut...",Anxiety disorder,Generalised,Homo sapiens


## 7. Resolve Tail (Disease) IDs and Names

1. Map MONDO IDs → MESH CUI via MedGen; update `tail` in-place.
2. Drop remaining MONDO rows (no MESH mapping found).
3. Derive `tail_detail_name` from DOID → DO label, else MESH → name dict, else supplementary MESH, else MONDO-MESH preferred name.
4. Re-assign `tail_id_is` after the MONDO→MESH conversion.

In [38]:
# Step 1 – MONDO → MESH conversion
final_df['tail'] = final_df['tail'].map(MONDO_2_MESH_dict).fillna(final_df['tail'])

# Step 2 – Drop rows still carrying MONDO IDs
before = len(final_df)
final_df = final_df[~final_df['tail'].str.startswith('MONDO')].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unmapped MONDO rows → {len(final_df):,} remaining")

# Step 3 – Derive tail_detail_name
final_df['tail_detail_name'] = final_df['tail'].apply(
    lambda x: DOID_disname_dict.get(x)
              if isinstance(x, str) and x.startswith('DOID')
              else (MESH_dict.get(x) or Mesh_supp_dict.get(x) or MONDOMESH_2_des_dict.get(x))
)

# Step 4 – Re-assign tail_id_is after MONDO→MESH swap
final_df['tail_id_is'] = np.where(
    final_df['tail'].isna(), None,
    np.where(final_df['tail'].str.startswith('DOID'), 'DOID', 'MESH')
)

print(f"Unique tail_id_is: {set(final_df['tail_id_is'])}")

Dropped 231 unmapped MONDO rows → 2,228,117 remaining
Unique tail_id_is: {'MESH', 'DOID'}


## 8. Deduplicate

Group by `(head, relation, tail)` and collapse `kg_source` with `::` separator.

In [39]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

print(f"Before deduplication: {len(final_df):,} rows")
final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
    
})

print(f"After deduplication: {len(final_df_dedup):,} rows")

Before deduplication: 2,228,117 rows
After deduplication: 1,565,657 rows


## 9. Resolve Head (Chemical) Names

1. PubChem IUPAC lookup for numeric CIDs.
2. Normalise DrugBank IDs to zero-padded 5-digit format.
3. DrugBank name fallback for `DB`-prefixed IDs still missing a name.
4. Drop rows where name could not be resolved for either node.

In [40]:
# Step 1 – PubChem lookup
final_df_dedup['head_detail_name'] = final_df_dedup['head'].astype(str).map(Pubchem_IUPAC_CID_Dict)
final_df_dedup['head_smiles_name'] = final_df_dedup['head'].astype(str).map(Pubchem_CID_Smile_Dict)

# Step 2 – Normalise DrugBank IDs
def standardize_drugbank_id(val):
    if isinstance(val, str):
        m = re.match(r'^DB(\d+)$', val)
        if m:
            return 'DB' + m.group(1).zfill(5)
    return val

final_df_dedup['head'] = final_df_dedup['head'].apply(standardize_drugbank_id)

# Step 3 – DrugBank fallback
mask = final_df_dedup['head_detail_name'].isna() & final_df_dedup['head'].str.startswith('DB')
final_df_dedup.loc[mask, 'head_detail_name'] = final_df_dedup.loc[mask, 'head'].map(Drugbank_dict)

# Step 4 – Drop unresolvable rows
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[
    ~final_df_dedup['head_detail_name'].isna() &
    ~final_df_dedup['tail_detail_name'].isna()
].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} unresolvable rows → {len(final_df_dedup):,} remaining")

Dropped 61,546 unresolvable rows → 1,504,111 remaining


## 10. Add Schema Columns and Enforce Column Order

In [41]:
# final_df_dedup['kg_type'] = 'generalised'
# final_df_dedup['species'] = 'Human'

# SMILES is extra — appended after REQUIRED_COLS
EXTRA_COLS = ['head_smiles_name']
final_df_dedup = final_df_dedup[REQUIRED_COLS + EXTRA_COLS]

print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (1504111, 14)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name
0,1000,ChemicalEntity_Disease,DOID:13413,ChemicalEntity,J,Disease,Hetionet,Pubchem,DOID,2-amino-1-phenylethanol,hepatic encephalopathy,Generalised,Homo sapiens,C1=CC=C(C=C1)C(CN)O
1,10000456,ChemicalEntity_Disease,DOID:0060180,ChemicalEntity,treat,Disease,TARKG,Pubchem,DOID,1'-[2-[4-(trifluoromethyl)phenyl]ethyl]spiro[1...,colitis,Generalised,Homo sapiens,C1CN(CCC12C3=CC=CC=C3NC(=O)O2)CCC4=CC=C(C=C4)C...
2,10000456,ChemicalEntity_Disease,DOID:10283,ChemicalEntity,treat,Disease,TARKG,Pubchem,DOID,1'-[2-[4-(trifluoromethyl)phenyl]ethyl]spiro[1...,prostate cancer,Generalised,Homo sapiens,C1CN(CCC12C3=CC=CC=C3NC(=O)O2)CCC4=CC=C(C=C4)C...
3,10000456,ChemicalEntity_Disease,DOID:10603,ChemicalEntity,treat,Disease,TARKG,Pubchem,DOID,1'-[2-[4-(trifluoromethyl)phenyl]ethyl]spiro[1...,glucose intolerance,Generalised,Homo sapiens,C1CN(CCC12C3=CC=CC=C3NC(=O)O2)CCC4=CC=C(C=C4)C...
4,10000456,ChemicalEntity_Disease,DOID:10763,ChemicalEntity,treat,Disease,TARKG,Pubchem,DOID,1'-[2-[4-(trifluoromethyl)phenyl]ethyl]spiro[1...,hypertension,Generalised,Homo sapiens,C1CN(CCC12C3=CC=CC=C3NC(=O)O2)CCC4=CC=C(C=C4)C...


In [42]:
print("\nkg_source values present:", set(final_df_dedup['kg_source']))


kg_source values present: {'DRKG', 'CROssBAR::DtiNet::pheknowlator', 'DtiNet::TARKG', 'DtiNet::TARKG::iBKH', 'CROssBAR::Hetionet::TARKG::iBKH::pheknowlator', 'DtiNet::Monarch_KG::iBKH', 'DtiNet::Hetionet::iBKH::pheknowlator', 'Hetionet', 'Hetionet::https://github.com/asgarhussain/phytochemicals/blob/main/phyto_chemicals.db::iBKH::pheknowlator', 'Hetionet::Monarch_KG', 'TARKG::pheknowlator', 'DtiNet::Hetionet::Monarch_KG::iBKH::pheknowlator', 'CROssBAR::TARKG::pheknowlator', 'Hetionet::TARKG', 'Hetionet::TARKG::iBKH::pheknowlator', 'Hetionet::TARKG::pheknowlator', 'DRKG::iBKH::pheknowlator', 'DRKG::DtiNet::TARKG::pheknowlator', 'TARKG::TTD::ttd', 'DtiNet::Hetionet::pheknowlator', 'https://github.com/asgarhussain/phytochemicals/blob/main/phyto_chemicals.db::iBKH', 'DtiNet::Monarch_KG::TARKG::iBKH::pheknowlator', 'CROssBAR::DtiNet::Hetionet::TARKG::iBKH::pheknowlator', 'Monarch_KG::https://github.com/asgarhussain/phytochemicals/blob/main/phyto_chemicals.db::pheknowlator', 'DRKG::TARKG::p

In [43]:
print("\nkg_source values present:", set(final_df_dedup['kg_type']))


kg_source values present: {'Generalised', 'Aging', 'Aging::Generalised'}


## 11. QC — NaN Check

In [44]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,842763,0,842763
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 12. Save Output

In [45]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 1,504,111 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CHEMICALENTITY_DISEASE/ALL_CHEMICALENTITY_DISEASE.csv


In [1]:
#

# Final Disease Mapping 

In [25]:
import pandas as pd

# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (

        df[detail_col]

        .astype(str)

        .str.lower()

        .str.strip()

        .map(mapping_dict)
    )

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),
        id_is_col
    ] = 'DOID'
    
    # everything else -> MESH
    df.loc[
        ~df[id_col].str.startswith('DOID:', na=False)
        &
        df[id_col].notna(),
        id_is_col
    ] = 'MESH'

    return df

In [26]:
# final_diseaseid_df[final_diseaseid_df['entity_name'] == 'age related macular degeneration 5']

In [27]:
# =========================================================
# LOAD KG FILE
# =========================================================

final_file_chem_dis = pd.read_csv(OUT_PATH)

# =========================================================
# MAP DISEASE IDS TO TAIL
# =========================================================

final_file_chem_dis = map_entity_ids(df=final_file_chem_dis,mapping_dict=disease_dict,side='tail')

# =========================================================
# RESULTS
# =========================================================
final_file_chem_dis

/tmp/ipykernel_64283/3444348868.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  final_file_chem_dis = pd.read_csv(OUT_PATH)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name,head_species,tail_species
0,1000,ChemicalEntity_Disease,DOID:13413,ChemicalEntity,J,Disease,Hetionet,Pubchem,DOID,2-amino-1-phenylethanol,hepatic encephalopathy,Generalised,Homo sapiens,C1=CC=C(C=C1)C(CN)O,Homo sapiens,Homo sapiens
1,10000456,ChemicalEntity_Disease,DOID:0060180,ChemicalEntity,treat,Disease,TARKG,Pubchem,DOID,1'-[2-[4-(trifluoromethyl)phenyl]ethyl]spiro[1...,colitis,Generalised,Homo sapiens,C1CN(CCC12C3=CC=CC=C3NC(=O)O2)CCC4=CC=C(C=C4)C...,Homo sapiens,Homo sapiens
2,10000456,ChemicalEntity_Disease,DOID:10283,ChemicalEntity,treat,Disease,TARKG,Pubchem,DOID,1'-[2-[4-(trifluoromethyl)phenyl]ethyl]spiro[1...,prostate cancer,Generalised,Homo sapiens,C1CN(CCC12C3=CC=CC=C3NC(=O)O2)CCC4=CC=C(C=C4)C...,Homo sapiens,Homo sapiens
3,10000456,ChemicalEntity_Disease,DOID:10603,ChemicalEntity,treat,Disease,TARKG,Pubchem,DOID,1'-[2-[4-(trifluoromethyl)phenyl]ethyl]spiro[1...,glucose intolerance,Generalised,Homo sapiens,C1CN(CCC12C3=CC=CC=C3NC(=O)O2)CCC4=CC=C(C=C4)C...,Homo sapiens,Homo sapiens
4,10000456,ChemicalEntity_Disease,DOID:10763,ChemicalEntity,treat,Disease,TARKG,Pubchem,DOID,1'-[2-[4-(trifluoromethyl)phenyl]ethyl]spiro[1...,hypertension,Generalised,Homo sapiens,C1CN(CCC12C3=CC=CC=C3NC(=O)O2)CCC4=CC=C(C=C4)C...,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1504106,DB15462,ChemicalEntity_Disease,C0596263,ChemicalEntity,GNBR::Pa::Compound:Disease,Disease,DRKG,Drugbank,MESH,Fucoxanthin,Carcinogenesis,Generalised,Homo sapiens,NaN,Homo sapiens,Homo sapiens
1504107,DB15528,ChemicalEntity_Disease,D009134,ChemicalEntity,DRUGBANK::treats::Compound:Disease,Disease,DRKG,Drugbank,MESH,Onasemnogene abeparvovec,"Muscular Atrophy, Spinal",Generalised,Homo sapiens,NaN,Homo sapiens,Homo sapiens
1504108,DB15591,ChemicalEntity_Disease,DOID:0060239,ChemicalEntity,GNBR::T::Compound:Disease,Disease,DRKG,Drugbank,DOID,Atrial natriuretic peptide,Van der Woude syndrome,Generalised,Homo sapiens,NaN,Homo sapiens,Homo sapiens
1504109,DB15593,ChemicalEntity_Disease,C030635,ChemicalEntity,DRUGBANK::treats::Compound:Disease,Disease,DRKG,Drugbank,MESH,Golodirsen,DMD circulating plasma factor,Generalised,Homo sapiens,NaN,Homo sapiens,Homo sapiens


In [31]:
final_file_chem_dis[final_file_chem_dis['tail'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species,head_smiles_name,head_species,tail_species


In [32]:
final_file_chem_dis.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file_chem_dis):,} rows → {OUT_PATH}")

Saved 1,504,111 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/CHEMICALENTITY_DISEASE/ALL_CHEMICALENTITY_DISEASE.csv
